# Week 3 Assignment · Fine-Tune an Analyst — Data, Discipline, Proof
### Build Custom AI — SarasAI · *ungraded practice*

The live session fine-tuned the Analyst end to end. Here you rebuild it yourself — **12 tasks
in four parts** — with the emphasis where professionals put it: the dataset and the eval,
not the trainer call.

| Part | Tasks | What you build |
|---|---|---|
| 1 · Baseline first | 1–3 | `chat()` helper · the format checker · the measured *before* |
| 2 · The dataset | 4–7 | your own generator · scale to 60 pairs · dedup · decontaminated split |
| 3 · Training | 8–9 | LoRA config with trainable-parameter accounting · the SFT run |
| 4 · Proof | 10–12 | tuned vs. base via `disable_adapter()` · a **blind win-rate judge** · save artifacts |

⏱ ~75–90 min including one short training run (a few minutes on a T4).
Self-checks follow most tasks; the solution notebook is for *checking*, not starting.

---
## Setup (given)

The 4-bit base model — QLoRA's foundation — plus the report contract:

In [ ]:
!pip install -q "transformers>=4.50" "trl>=0.17" "peft>=0.14" bitsandbytes datasets accelerate sentence-transformers

In [ ]:
import torch

assert torch.cuda.is_available(), "This assignment needs a GPU (T4 is enough)."
BF16_OK = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print("dtype:", DTYPE)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE_ID)
base = AutoModelForCausalLM.from_pretrained(BASE_ID, quantization_config=bnb, device_map="auto")
print(f"4-bit footprint: {base.get_memory_footprint()/1e9:.2f} GB   (fp32 would be ~6 GB)")

In [ ]:
REPORT_FORMAT = """## ROOT CAUSE ANALYSIS
**Product:** <model id>
**Defect:** <one-line summary>
**Severity:** LOW | MEDIUM | HIGH
**Evidence:** <cited facts from the input>
**Probable cause:** <one short paragraph>
**Recommended action:** <one concrete step>"""

def make_input(defect_note, evidence):
    return (f"Defect note from visual inspection:\n{defect_note}\n\n"
            f"Retrieved evidence:\n{evidence}\n\n"
            f"Write a root cause analysis report.")

---
## Part 1 · Baseline first — never train before you've measured *before*

### ✏️ Task 1 — the `chat()` helper

> 💡 **Hint:** third time you've written this helper this course — that's on purpose. It should be automatic now.

In [ ]:
# ╔══════════════════════ TASK 1 ══════════════════════╗
# the `chat()` helper
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
def chat(model, user_text, max_new_tokens=300):
    """One greedy chat turn. Same recipe as Week 2 — write it from memory if you can."""
    # apply_chat_template (add_generation_prompt, return_dict, "pt") -> generate -> decode
    # only the NEW tokens (slice off the prompt), skip special tokens, strip.
    inputs = ...
    out = ...
    return ...

print(chat(base, "Say READY.", max_new_tokens=5))

### ✏️ Task 2 — the format checker — your deterministic metric

> 💡 **Hint:** all(h in text for h in REQUIRED) + one regex for the severity value.

In [ ]:
# ╔══════════════════════ TASK 2 ══════════════════════╗
# the format checker — your deterministic metric
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
import re

REQUIRED = ["## ROOT CAUSE ANALYSIS", "**Product:**", "**Defect:**", "**Severity:**",
            "**Evidence:**", "**Probable cause:**", "**Recommended action:**"]

def format_ok(text):
    """True iff ALL seven headers are present AND Severity is exactly LOW/MEDIUM/HIGH."""
    return ...

def format_rate(outputs):
    """Fraction of outputs passing format_ok."""
    return ...

# truth table
tests = [REPORT_FORMAT.replace("LOW | MEDIUM | HIGH", "HIGH"),      # valid
         "## ROOT CAUSE ANALYSIS\n**Product:** X",                   # missing sections
         REPORT_FORMAT.replace("LOW | MEDIUM | HIGH", "CRITICAL")]  # invalid severity
for t in tests:
    print(format_ok(t), "<-", t[:60].replace("\n", " "))

In [ ]:
# ── self-check ──
assert format_ok(REPORT_FORMAT.replace("LOW | MEDIUM | HIGH", "MEDIUM"))
assert not format_ok("just some text")
assert not format_ok(REPORT_FORMAT.replace("LOW | MEDIUM | HIGH", "SEVERE"))
print("✅ checker is strict on severity AND completeness — mechanical metrics first, always")

### ✏️ Task 3 — measure the base model on the task

> 💡 **Hint:** even WITH the format in the prompt, small models drift — renamed headers, free-texted severity. That gap is what training closes.

In [ ]:
# ╔══════════════════════ TASK 3 ══════════════════════╗
# measure the base model on the task
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
example = make_input(
    "Cracked lid hinge observed on 4 units of KT-2000 kettle, batch B-1201.",
    "- review_041: 'lid cracked after one week'\n- spec: lid part PP-114, polypropylene",
)

# (a) ask the base model to write a report for `example`, WITH the format spec in the prompt:
#     prompt = example + "\n\nUse exactly this format:\n" + REPORT_FORMAT
baseline = ...
print(baseline[:500])

# (b) does it pass?
print("\nformat_ok:", ...)

---
## Part 2 · The dataset — where fine-tunes are won and lost

### ✏️ Task 4 — your generator — different vocabulary than the lecture, on purpose

> 💡 **Hint:** vary NOUNS (≥4/≥8/≥5 vocab) AND SENTENCE STRUCTURE (2-3 note templates) — embedding dedup measures both; single-template inputs collapse into near-duplicates.

In [ ]:
# ╔══════════════════════ TASK 4 ══════════════════════╗
# your generator — different vocabulary than the lecture, on purpose
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
import itertools, random
random.seed(7)                      # your seed — NOT the class's 42

# ≥4 products, ≥8 (defect, severity) pairs, ≥5 causes. INVENT YOUR OWN — if you copy the
# lecture's, your model may look artificially good later (why? -> leakage across notebooks).
products = ["WM-3 washing machine", ...]
defects  = [("door seal leak", "MEDIUM"), ...]
causes   = ["gasket compound out of spec", ...]

def synth_pair(product, defect, severity, cause, idx):
    """{"input": make_input(...), "output": <report following REPORT_FORMAT exactly>}"""
    note = ...                       # 1 sentence: defect, N units, product, batch id
    evidence = ...                   # 2-3 "- source: fact" lines; vary with severity
    report = ...                     # all seven sections, severity from the tuple
    return {"input": make_input(note, evidence), "output": report}

pair = synth_pair(products[0], defects[0][0], defects[0][1], causes[0], 0)
print(pair["output"])

In [ ]:
# ── self-check: one pair, fully compliant ──
assert format_ok(pair["output"]), "your synthetic report must pass YOUR OWN format checker"
assert len(products) >= 4 and len(defects) >= 8 and len(causes) >= 5, "more diversity needed"
print("✅ generator produces compliant reports from a diverse vocabulary")

### ✏️ Task 5 — scale to 60 raw pairs

> 💡 **Hint:** itertools.product gives (product, (defect, severity), cause) tuples — unpack carefully.

In [ ]:
# ╔══════════════════════ TASK 5 ══════════════════════╗
# scale to 60 raw pairs
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
# Cartesian product of (products x defects x causes), shuffled, first 60 -> synth_pair each.
combos = ...
...
raw_rows = ...

print(len(raw_rows), "raw pairs")
bad = [i for i, r in enumerate(raw_rows) if not format_ok(r["output"])]
print("non-compliant rows:", bad or "none")

### ✏️ Task 6 — embedding dedup

> 💡 **Hint:** near-duplicates teach memorization, not generalization — np.dot on normalized vectors is cosine.

In [ ]:
# ╔══════════════════════ TASK 6 ══════════════════════╗
# embedding dedup
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
SIM = 0.90                          # one threshold for BOTH hygiene checks

X = embedder.encode([r["input"] for r in raw_rows], normalize_embeddings=True)

# keep a row only if its max cosine vs all already-kept rows is <= SIM
keep, kept_vecs = [], []
for i, row in enumerate(raw_rows):
    ...

print(f"dedup: {len(raw_rows)} -> {len(keep)} rows")

### ✏️ Task 7 — split, then DECONTAMINATE — an action, not a printout

> 💡 **Hint:** X_train @ X_test.T — then a list comprehension keeping rows with max sim <= SIM.

In [ ]:
# ╔══════════════════════ TASK 7 ══════════════════════╗
# split, then DECONTAMINATE — an action, not a printout
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
# (a) first 10 kept rows -> test (frozen 🔒), rest -> train
test_rows, train_rows = ...

# (b) embed both sides, build the (n_train, n_test) cosine matrix
X_test  = ...
X_train = ...
sims = ...

# (c) DROP every train row whose max similarity to ANY test row exceeds SIM
train_rows = ...

print(f"train: {len(train_rows)}   test: {len(test_rows)} 🔒")

In [ ]:
# ── self-check: certify the split ──
assert len(test_rows) == 10 and len(train_rows) >= 15, "too few rows survived — vary sentence STRUCTURE (multiple note templates), not just nouns"
X_tr = embedder.encode([r["input"] for r in train_rows], normalize_embeddings=True)
X_te = embedder.encode([r["input"] for r in test_rows], normalize_embeddings=True)
leak = float((X_tr @ X_te.T).max())
assert leak <= 0.90 + 1e-6, f"leakage! max train-test similarity {leak:.3f}"
print(f"✅ split certified clean — max cross-similarity {leak:.3f}")
print("(0 rows dropped is normal here: dedup at the same threshold already guarantees it — the habit is the lesson)")
print("Honest caveat: test rows come from the SAME generator as training, so metrics will")
print("flatter you. Your capstone teacher-model data won't be this easy — the discipline is the lesson.")

---
## Part 3 · Training

### ✏️ Task 8 — LoRA config + the parameter-count sanity check

> 💡 **Hint:** p.numel() sums; p.requires_grad marks the adapters. Expect ~1-2% trainable.

In [ ]:
# ╔══════════════════════ TASK 8 ══════════════════════╗
# LoRA config + the parameter-count sanity check
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_cfg = LoraConfig(
    r=...,                          # 8-32 is sane for a format task
    lora_alpha=...,                 # rule of thumb: 2 * r
    target_modules=[...],           # Qwen2.5 attention + MLP:
                                    #   q_proj k_proj v_proj o_proj gate_proj up_proj down_proj
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)

model = get_peft_model(prepare_model_for_kbit_training(base), lora_cfg)

# compute trainable vs total parameters yourself (don't use print_trainable_parameters):
trainable = ...
total = ...
print(f"trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B  ({100*trainable/total:.2f}%)")

In [ ]:
# ── self-check ──
assert trainable / total < 0.05, "more than 5% trainable? your target_modules or r is off"
assert trainable > 0, "nothing trainable — did you call get_peft_model?"
print("✅ that tiny sliver is ALL we train — the 4-bit base stays frozen")

### ✏️ Task 9 — SFT config + train

> 💡 **Hint:** watch the loss: fast first-epoch drop = learning the format. Loss ≈ 0 = memorizing — back off (smaller r / fewer epochs / more data).

In [ ]:
# ╔══════════════════════ TASK 9 ══════════════════════╗
# SFT config + train
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list([
    {"messages": [{"role": "user", "content": r["input"]},
                  {"role": "assistant", "content": r["output"]}]} for r in train_rows])

sft_cfg = SFTConfig(
    output_dir="asg-analyst",
    num_train_epochs=...,                       # 2-3
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,              # effective batch = 8
    learning_rate=...,                          # LoRA tolerates 1e-4 .. 3e-4
    lr_scheduler_type="cosine", logging_steps=5,
    bf16=..., fp16=...,                         # match the GPU (you have BF16_OK)
    optim="paged_adamw_8bit",                   # QLoRA trick #3
    max_length=1024, report_to="none",
)

trainer = SFTTrainer(model=model, args=sft_cfg, train_dataset=train_ds, processing_class=tok)
trainer.train()

---
## Part 4 · Proof

### ✏️ Task 10 — tuned vs. base — the `disable_adapter()` A/B

> 💡 **Hint:** model.disable_adapter() is a context manager — the session's 'trap' slide.

In [ ]:
# ╔══════════════════════ TASK 10 ══════════════════════╗
# tuned vs. base — the `disable_adapter()` A/B
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
test_inputs = [r["input"] for r in test_rows]

# (a) TUNED outputs — the model as it now is
tuned_out = ...

# (b) BASE outputs — careful: adapters were injected INTO base's module tree, so
#     chat(base, ...) would give TUNED outputs. Use the context manager.
with ...:
    base_out = ...

print(f"format adherence — base: {format_rate(base_out):.0%}   tuned: {format_rate(tuned_out):.0%}")

### ✏️ Task 11 — a blind win-rate judge — with randomized A/B order

> 💡 **Hint:** flip a coin per pair; a tuned win = judge picked the slot the tuned output sat in.

In [ ]:
# ╔══════════════════════ TASK 11 ══════════════════════╗
# a blind win-rate judge — with randomized A/B order
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
def judge_pair(inp, out_a, out_b):
    """Ask the model (adapters OFF) which report is better. Returns 'A' or 'B'."""
    prompt = (f"Task input:\n{inp[:400]}\n\nReport A:\n{out_a[:400]}\n\nReport B:\n{out_b[:400]}\n\n"
              "Which report is better structured, better grounded in the input, and more "
              "actionable? Reply with exactly one letter: A or B.")
    with model.disable_adapter():
        v = chat(model, prompt, max_new_tokens=3)
    return "A" if v.strip().upper()[:1] == "A" else "B"

wins = 0
for r, t_out, b_out in zip(test_rows, tuned_out, base_out):
    # randomize which side is tuned (why? position bias — judges favor one slot).
    # Track where the tuned output went, count a win when the judge picks it.
    ...

print(f"win-rate (tuned vs base): {wins}/{len(test_rows)}")

In [ ]:
# ── honest caveat (given — read it) ──
print("""Judge caveats, same as Week 2's — plus one new one:
 1. The judge here is the BASE model judging its own family -> bias. Capstone: stronger judge.
 2. Win-rate on 10 pairs moves +/-10% per flipped verdict. Report the count (7/10), not just %.
 3. You randomized A/B order — that wasn't optional. Fixed order can shift win-rates by 10-20%
    on small models. You just implemented the fix most tutorials skip.""")

### ✏️ Task 12 — save the artifacts Week 4 needs

> 💡 **Hint:** model.save_pretrained(dir) + json.dump — the whole fine-tune fits in tens of MB.

In [ ]:
# ╔══════════════════════ TASK 12 ══════════════════════╗
# save the artifacts Week 4 needs
# Replace every `...` below. Run the self-check cell after.
# ╚══════════════════════════════════════════════════════╝
import json

# (a) save the adapter to "analyst-lora-adapter"
...

# (b) save the frozen test set to "week3_test_rows.json" (Week 4 re-scores it after compression)
...

import os
size = sum(os.path.getsize(os.path.join("analyst-lora-adapter", f))
           for f in os.listdir("analyst-lora-adapter")) / 1e6
print(f"adapter: {size:.0f} MB   test set: week3_test_rows.json ({len(test_rows)} rows)")

In [ ]:
# ── self-check ──
import os
assert os.path.isdir("analyst-lora-adapter"), "adapter not saved"
assert os.path.exists("week3_test_rows.json"), "frozen test set not saved"
print("✅ both artifacts on disk — Week 4's assignment will pick them up automatically")

---
## Wrap-up · What you practiced

| Task | Skill | Where it goes next |
|---|---|---|
| 1–3 | chat helper, deterministic metric, measured baseline | every eval you'll ever run |
| 4–7 | diverse generation, dedup, decontamination | the graded dataset card |
| 8–9 | LoRA's four numbers, parameter accounting, SFT | your knobs when quality disappoints |
| 10–12 | disable_adapter A/B, blind randomized win-rate, artifacts | Week 4 re-uses all three |

**Reflection (2 min):** your tuned format rate is probably ~100% on same-generator test rows.
What specifically would make this test honest — and how does the graded increment's
teacher-model data provide it?

*Ungraded — nothing to submit. Your saved adapter + frozen set feed directly into Week 4's
assignment.*